In [4]:

import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, precision_score, recall_score

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

print(ratings.head())

user_item_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

user_item_matrix_filled = user_item_matrix.fillna(0)

print(user_item_matrix_filled.shape)

kmeans = KMeans(n_clusters=5, random_state=42)

user_clusters = kmeans.fit_predict(user_item_matrix_filled)

user_cluster_df = pd.DataFrame({
    "userId": user_item_matrix_filled.index,
    "cluster": user_clusters
})

ratings = ratings.merge(user_cluster_df, on="userId")

X = ratings[['userId','movieId','cluster']]
y = ratings['rating']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
gbr = GradientBoostingRegressor()

gbr.fit(X_train, y_train)

gbr_pred = gbr.predict(X_test)
nn = MLPRegressor(
    hidden_layer_sizes=(128,64),
    max_iter=50,
    random_state=42
)

nn.fit(X_train, y_train)

nn_pred = nn.predict(X_test)
rmse_gbr = np.sqrt(mean_squared_error(y_test, gbr_pred))
rmse_nn = np.sqrt(mean_squared_error(y_test, nn_pred))

print("Gradient Boosting RMSE:", rmse_gbr)
print("Neural Network RMSE:", rmse_nn)
threshold = 4

y_test_binary = (y_test >= threshold).astype(int)

gbr_binary = (gbr_pred >= threshold).astype(int)
nn_binary = (nn_pred >= threshold).astype(int)

precision_gbr = precision_score(y_test_binary, gbr_binary)
recall_gbr = recall_score(y_test_binary, gbr_binary)

precision_nn = precision_score(y_test_binary, nn_binary)
recall_nn = recall_score(y_test_binary, nn_binary)

print("GBR Precision:", precision_gbr)
print("GBR Recall:", recall_gbr)

print("NN Precision:", precision_nn)
print("NN Recall:", recall_nn)
def recommend_movies(user_id, model, top_n=5):

    user_cluster = user_cluster_df[user_cluster_df.userId==user_id]['cluster'].values[0]

    movies_list = movies['movieId'].unique()

    data = pd.DataFrame({
        'userId':[user_id]*len(movies_list),
        'movieId':movies_list,
        'cluster':[user_cluster]*len(movies_list)
    })

    predictions = model.predict(data)

    data['predicted_rating'] = predictions

    top_movies = data.sort_values(
        'predicted_rating',
        ascending=False
    ).head(top_n)

    return top_movies

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
(610, 9724)
Gradient Boosting RMSE: 0.9834670718287042
Neural Network RMSE: 19.388897972935343
GBR Precision: 0.8176795580110497
GBR Recall: 0.04581570529357135
NN Precision: 0.5
NN Recall: 0.0004127541017438861
